# Clustering / embedding diagnostics

Loads a trained encoder, extracts globally-pooled deepest-stage
embeddings on the validation set + the canonical sanity batch,
then reports collapse / cluster diagnostics and produces 2D
scatter plots. Read-only with respect to the encoder weights.


## Configuration


In [ ]:
from types import SimpleNamespace
cluster_cfg = SimpleNamespace(
    seed              = 42,
    encoder_ckpt      = '../outputs/REPLACE_ME/best_model.pt',  # path to a checkpoint
    output_dir        = '../outputs/clustering_check',
    data_root         = '../../../data/patches_128',
    exclude_patterns  = ['KONTROLA'],
    val_split         = 0.10,
    batch_size        = 64,
    num_workers       = 2,
    pin_memory        = True,
    max_batches       = 32,            # how many val batches to embed
    n_clusters        = 8,
    reduction         = 'auto',        # 'auto' | 'umap' | 'pca'
    channel_names     = ['pre_synaptic', 'post_synaptic', 'structural'],
)


## Imports


In [ ]:
import os, sys, json, time
from datetime import datetime
from pathlib import Path


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt


## Path setup


In [ ]:
NB_DIR = Path.cwd().resolve()
NB_NEW = NB_DIR.parent if NB_DIR.parent.name == 'notebooks' else NB_DIR.parents[1]
ROOT   = NB_NEW.parent
for p in (NB_NEW, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('NB_NEW:', NB_NEW); print('ROOT  :', ROOT)


In [ ]:
from training.config import ModelCfg
from training.seeding import seed_everything
from training.logging import setup_logger
from training.augment import ValSingleViewTransform
from training.sanity_batch import SANITY_TRAIN_INDICES, SANITY_VAL_INDICES, fixed_single_view_batch
from training.viz import plot_two_views, plot_embedding_2d
from models.swin import build_swin_encoder, count_params
from clustering.core import extract_pooled_embeddings, effective_rank, mean_pairwise_cos, reduce_2d
from training.data import TransformedSubset
from data.patch_dataset import PatchDataset


## Output directory + logger


In [ ]:
RUN_TS  = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = Path(cluster_cfg.output_dir) / f'cluster_{RUN_TS}'
save_dir.mkdir(parents=True, exist_ok=True)
logger = setup_logger('clustering', save_dir / 'run.log')
logger.info(f'encoder checkpoint = {cluster_cfg.encoder_ckpt}')
logger.info(f'save_dir           = {save_dir}')


## Seed and device


In [ ]:
_ = seed_everything(cluster_cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'device = {device}')


## Load checkpoint metadata + build encoder


In [ ]:
ckpt_path = Path(cluster_cfg.encoder_ckpt)
if not ckpt_path.exists():
    raise FileNotFoundError(f'encoder checkpoint not found: {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
logger.info(f'ckpt keys = {sorted(ckpt.keys())}')


In [ ]:
model_cfg = ModelCfg()  # defaults match the training templates
encoder = build_swin_encoder(model_cfg).to(device)
encoder.load_state_dict(ckpt['encoder_state_dict'])
encoder.eval()
logger.info(f'encoder params = {count_params(encoder) / 1e6:.2f} M')


## Channel statistics from the checkpoint


In [ ]:
ch_mean = torch.tensor(ckpt['channel_mean']) if 'channel_mean' in ckpt else None
ch_std  = torch.tensor(ckpt['channel_std'])  if 'channel_std'  in ckpt else None
if ch_mean is None or ch_std is None:
    logger.warning('checkpoint has no channel_mean/std -- recomputing from the data')
    from training.data import compute_channel_stats
    raw_tmp = PatchDataset(root=cluster_cfg.data_root, exclude_patterns=cluster_cfg.exclude_patterns)
    ch_mean, ch_std = compute_channel_stats(raw_tmp, in_channels=model_cfg.in_channels, max_samples=2048)
logger.info(f'ch_mean = {ch_mean.tolist()}')
logger.info(f'ch_std  = {ch_std.tolist()}')


## Build val dataset / dataloader


In [ ]:
raw = PatchDataset(root=cluster_cfg.data_root, exclude_patterns=cluster_cfg.exclude_patterns)
n_val   = int(len(raw) * cluster_cfg.val_split)
n_train = len(raw) - n_val
g       = torch.Generator().manual_seed(cluster_cfg.seed)
train_subset, val_subset = random_split(raw, [n_train, n_val], generator=g)
val_transform = ValSingleViewTransform(ch_mean, ch_std)
val_ds        = TransformedSubset(val_subset, val_transform)
val_loader    = DataLoader(
    val_ds, batch_size=cluster_cfg.batch_size, shuffle=False,
    num_workers=cluster_cfg.num_workers, pin_memory=cluster_cfg.pin_memory,
    persistent_workers=cluster_cfg.num_workers > 0,
)
logger.info(f'val patches = {len(val_ds)}')


## Sanity batch visualisation (canonical indices)


In [ ]:
x_sanity, sanity_idx = fixed_single_view_batch(
    val_subset, SANITY_VAL_INDICES, val_transform, seed=cluster_cfg.seed + 21,
)
logger.info(f'sanity val indices = {sanity_idx}  shape = {tuple(x_sanity.shape)}')


## Extract pooled embeddings on the val set


In [ ]:
Z_val = extract_pooled_embeddings(
    encoder, val_loader, device=device, max_batches=cluster_cfg.max_batches,
)
logger.info(f'val embeddings: N={Z_val.shape[0]}  D={Z_val.shape[1]}')


## Embeddings on the canonical sanity batch (separate, non-shuffled)


In [ ]:
with torch.no_grad():
    z = encoder(x_sanity.to(device).contiguous())[-1]
    Z_sanity = z.mean(dim=(2, 3)).cpu()
logger.info(f'sanity embeddings: {tuple(Z_sanity.shape)}')


## Collapse diagnostics


In [ ]:
eff, r_max, S = effective_rank(Z_val)
mc            = mean_pairwise_cos(Z_val)
logger.info(f'effective rank = {eff:.2f} / {r_max} ({100 * eff / max(r_max, 1):.1f}%)')
logger.info(f'mean pairwise cosine = {mc:.4f}  ({"COLLAPSED" if mc > 0.95 else "OK"})')
metrics = {'eff_rank': eff, 'r_max': r_max, 'mean_cos': mc,
           'N': int(Z_val.shape[0]), 'D': int(Z_val.shape[1])}
(save_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))


## 2D projection (UMAP if available, else PCA)


In [ ]:
Z2_val, used = reduce_2d(Z_val, method=cluster_cfg.reduction, seed=cluster_cfg.seed)
logger.info(f'reduction used: {used}')


## K-means clustering on the val embeddings


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
km = KMeans(n_clusters=cluster_cfg.n_clusters, n_init=10, random_state=cluster_cfg.seed)
labels = km.fit_predict(Z_val.numpy())
sil = silhouette_score(Z_val.numpy(), labels) if cluster_cfg.n_clusters > 1 else float('nan')
logger.info(f'k-means k={cluster_cfg.n_clusters}  silhouette={sil:.4f}')
metrics['silhouette'] = float(sil)
metrics['n_clusters'] = int(cluster_cfg.n_clusters)
(save_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))


## Plot 2D scatter coloured by cluster


In [ ]:
_ = plot_embedding_2d(
    Z2_val, labels=labels,
    title=f'Val embeddings ({used.upper()}) -- k-means k={cluster_cfg.n_clusters}',
    singular_values=S,
    save_to=save_dir / 'val_embedding_kmeans.png',
)
plt.show()


## Sanity-batch embeddings overlaid on the val scatter


In [ ]:
Z_combined = torch.cat([Z_val, Z_sanity], dim=0)
Z2_comb, _ = reduce_2d(Z_combined, method=cluster_cfg.reduction, seed=cluster_cfg.seed)
n = Z_val.shape[0]
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
ax.scatter(Z2_comb[:n, 0], Z2_comb[:n, 1], s=5, alpha=0.4, label='val')
ax.scatter(Z2_comb[n:, 0], Z2_comb[n:, 1], s=80, c='red', marker='x', label='sanity batch')
ax.legend(loc='best')
ax.set_xlabel('dim 1'); ax.set_ylabel('dim 2')
ax.set_title('Val + canonical sanity batch (overlaid)')
fig.tight_layout()
fig.savefig(save_dir / 'sanity_overlay.png', dpi=120, bbox_inches='tight')
plt.show()


## Save embeddings + labels for downstream use


In [ ]:
np.save(save_dir / 'val_embeddings.npy',  Z_val.numpy())
np.save(save_dir / 'val_labels.npy',      labels)
np.save(save_dir / 'sanity_embeddings.npy', Z_sanity.numpy())
logger.info(f'embeddings + labels written to {save_dir}')
